In [ ]:
# CELL 1 — Install dependencies
!pip install einops yfinance

In [ ]:
# CELL 2 — Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm
from einops import rearrange

In [ ]:
# CELL 3 — Download 10-asset Yahoo Finance OHLCV dataset
tickers = ["AAPL","MSFT","GOOG","AMZN","TSLA","NVDA","META","SPY","GLD","BTC-USD"]

df = yf.download(tickers, start="2010-01-01", auto_adjust=False)
df = df.ffill().dropna()

data = df.values.astype(np.float32)
print("Full dataset shape (T, 60):", data.shape)

[*********************100%***********************]  10 of 10 completed

Full dataset shape (T, 60): (4124, 60)


In [ ]:
# CELL 4 — Normalize dataset (per feature, critical)
mean = data.mean(axis=0, keepdims=True)
std = data.std(axis=0, keepdims=True) + 1e-6

data = (data - mean) / std

print("Normalized data mean (first 5):", data.mean(axis=0)[:5])
print("Normalized data std  (first 5):", data.std(axis=0)[:5])

Normalized data mean (first 5): [-5.919996e-08  2.959998e-08  2.959998e-08 -1.479999e-07 -5.919996e-08]
Normalized data std  (first 5): [1. 1. 1. 1. 1.]


In [ ]:
# CELL 5 — 80/10/10 train/val/test split
N = len(data)
train_end = int(N * 0.8)
val_end = int(N * 0.9)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print("Train:", train_data.shape)
print("Val:  ", val_data.shape)
print("Test: ", test_data.shape)

Train: (3299, 60)
Val:   (412, 60)
Test:  (413, 60)


In [ ]:
# CELL 6 — Windowed dataset class
class WindowDataset(Dataset):
    def __init__(self, series, input_len=96, pred_len=24):
        self.series = series.astype(np.float32)
        self.input_len = input_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.series) - self.input_len - self.pred_len

    def __getitem__(self, idx):
        x = self.series[idx : idx + self.input_len]  # [L, D]
        y = self.series[idx + self.input_len : idx + self.input_len + self.pred_len]  # [pred_len, D]
        return torch.tensor(x), torch.tensor(y)

In [ ]:
# CELL 7 — Create datasets and dataloaders
input_len = 96
pred_len = 24
batch_size = 16  # lower batch_size for big model

train_ds = WindowDataset(train_data, input_len, pred_len)
val_ds = WindowDataset(val_data, input_len, pred_len)
test_ds = WindowDataset(test_data, input_len, pred_len)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size)
test_dl = DataLoader(test_ds, batch_size=batch_size)

print("Train batches:", len(train_dl))
print("Val batches:  ", len(val_dl))
print("Test batches: ", len(test_dl))

Train batches: 199
Val batches:   19
Test batches:  19


In [ ]:
# CELL 8 — Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: [B, L, d_model]
        L = x.size(1)
        return x + self.pe[:, :L, :]

In [ ]:
# CELL 9 — Fourier mixing block
class FourierBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.proj_in = nn.Linear(d_model, d_model)
        self.proj_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: [B, L, d_model]
        x_res = x
        x = self.proj_in(x)
        x_ft = torch.fft.rfft(x, dim=1)
        x_ft = x_ft * 0.5
        x = torch.fft.irfft(x_ft, n=x.size(1), dim=1)
        x = self.proj_out(x)
        x = self.dropout(x)
        return x + x_res

In [ ]:
# CELL 10 — FEDformer-style encoder and decoder layers
class FEDformerEncoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.fourier_block = FourierBlock(d_model, dropout=dropout)
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, src):
        # src: [B, L, d_model]
        attn_out, _ = self.self_attn(src, src, src)
        src = self.norm1(src + self.dropout(attn_out))

        fourier_out = self.fourier_block(src)
        src = self.norm2(src + self.dropout(fourier_out))

        ff = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = self.norm3(src + self.dropout(ff))
        return src


class FEDformerDecoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.fourier_block = FourierBlock(d_model, dropout=dropout)
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.norm4 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, tgt, memory):
        # tgt:    [B, L_t, d_model]
        # memory: [B, L_s, d_model]
        attn_out, _ = self.self_attn(tgt, tgt, tgt)
        tgt = self.norm1(tgt + self.dropout(attn_out))

        cross_out, _ = self.cross_attn(tgt, memory, memory)
        tgt = self.norm2(tgt + self.dropout(cross_out))

        fourier_out = self.fourier_block(tgt)
        tgt = self.norm3(tgt + self.dropout(fourier_out))

        ff = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = self.norm4(tgt + self.dropout(ff))
        return tgt

In [ ]:
# CELL 11 — Full FEDformer-style model (large, encoder-decoder)
class FEDformerLarge(nn.Module):
    def __init__(
        self,
        input_dim,
        d_model=512,
        n_heads=8,
        d_ff=2048,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dropout=0.1,
        input_len=96,
        pred_len=24,
    ):
        super().__init__()
        self.input_dim = input_dim      # 60
        self.d_model = d_model
        self.input_len = input_len
        self.pred_len = pred_len

        self.enc_in = nn.Linear(input_dim, d_model)
        self.dec_in = nn.Linear(input_dim, d_model)

        self.pos_enc = PositionalEncoding(d_model, max_len=max(input_len, pred_len) + 10)

        self.encoder_layers = nn.ModuleList([
            FEDformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])

        self.decoder_layers = nn.ModuleList([
            FEDformerDecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])

        self.proj_out = nn.Linear(d_model, input_dim)

    def forward(self, x_enc, x_dec):
        # x_enc: [B, input_len, input_dim]
        # x_dec: [B, pred_len, input_dim]
        B, L_enc, _ = x_enc.shape
        B2, L_dec, _ = x_dec.shape
        assert B == B2

        enc = self.enc_in(x_enc)
        enc = self.pos_enc(enc)
        for layer in self.encoder_layers:
            enc = layer(enc)

        dec = self.dec_in(x_dec)
        dec = self.pos_enc(dec)
        for layer in self.decoder_layers:
            dec = layer(dec, enc)

        out = self.proj_out(dec)  # [B, pred_len, input_dim]
        return out

In [ ]:
# CELL 12 — Initialize model, optimizer, loss, metrics
device = "cuda" if torch.cuda.is_available() else "cpu"
feature_dim = train_data.shape[1]  # 60

model = FEDformerLarge(
    input_dim=feature_dim,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dropout=0.1,
    input_len=input_len,
    pred_len=pred_len,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # smaller LR for big model
criterion = nn.MSELoss()

def mae(pred, true):
    return torch.mean(torch.abs(pred - true))

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {num_params:,}")

FEDformerLarge(
  (enc_in): Linear(in_features=60, out_features=512, bias=True)
  (dec_in): Linear(in_features=60, out_features=512, bias=True)
  (pos_enc): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-3): 4 x FEDformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (fourier_block): FourierBlock(
        (proj_in): Linear(in_features=512, out_features=512, bias=True)
        (proj_out): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (linear1): Linear(in_features=512, out_features=2048, bias=True)
      (linear2): Linear(in_features=2048, out_features=512, bias=True)
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout): Dr

In [ ]:
# CELL 13 — Training and validation functions
def train_epoch():
    model.train()
    total_loss = 0.0

    for x, y in tqdm(train_dl, desc="Training", leave=False):
        x, y = x.to(device), y.to(device)  # x: [B, 96, 60], y: [B, 24, 60]

        # decoder input: last pred_len steps of the encoder input (or zeros)
        dec_in = torch.zeros_like(y)  # teacher forcing style
        # you could also use y shifted, or last part of x; for now zeros

        optimizer.zero_grad()
        pred = model(x, dec_in)       # [B, 24, 60]
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_dl)


def validate_epoch():
    model.eval()
    mse_total = 0.0
    mae_total = 0.0

    with torch.no_grad():
        for x, y in tqdm(val_dl, desc="Validating", leave=False):
            x, y = x.to(device), y.to(device)
            dec_in = torch.zeros_like(y)
            pred = model(x, dec_in)

            mse_total += criterion(pred, y).item()
            mae_total += mae(pred, y).item()

    return mse_total / len(val_dl), mae_total / len(val_dl)

In [ ]:
# CELL 14 — Run training for 10 epochs
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    train_loss = train_epoch()
    val_mse, val_mae = validate_epoch()

    print(f"Train Loss: {train_loss:.6f}")
    print(f"Val MSE:    {val_mse:.6f}")
    print(f"Val MAE:    {val_mae:.6f}")


===== Epoch 1/10 =====


Train Loss: 0.170819
Val MSE:    0.393296
Val MAE:    0.495781

===== Epoch 2/10 =====


Train Loss: 0.129244
Val MSE:    0.486282
Val MAE:    0.577247

===== Epoch 3/10 =====


Train Loss: 0.114818
Val MSE:    0.457266
Val MAE:    0.552515

===== Epoch 4/10 =====


Train Loss: 0.105418
Val MSE:    0.450627
Val MAE:    0.557165

===== Epoch 5/10 =====


...

Train Loss: 0.087800
Val MSE:    0.521342
Val MAE:    0.608657

===== Epoch 7/10 =====


Train Loss: 0.081429
Val MSE:    0.465429
Val MAE:    0.561497

===== Epoch 8/10 =====


Train Loss: 0.076053
Val MSE:    0.424341
Val MAE:    0.531274

===== Epoch 9/10 =====


Train Loss: 0.070335
Val MSE:    0.430685
Val MAE:    0.535329

===== Epoch 10/10 =====


Train Loss: 0.065985
Val MSE:    0.460748
Val MAE:    0.559503


In [ ]:
# CELL 15 — Final test evaluation
def evaluate_test():
    model.eval()
    mse_total = 0.0
    mae_total = 0.0

    with torch.no_grad():
        for x, y in tqdm(test_dl, desc="Testing"):
            x, y = x.to(device), y.to(device)
            dec_in = torch.zeros_like(y)
            pred = model(x, dec_in)

            mse_total += criterion(pred, y).item()
            mae_total += mae(pred, y).item()

    return mse_total / len(test_dl), mae_total / len(test_dl)


test_mse, test_mae = evaluate_test()
print("\n===== FINAL TEST RESULTS (FEDformer-Large) =====")
print(f"Test MSE: {test_mse:.6f}")
print(f"Test MAE: {test_mae:.6f}")

Testing: 100%|██████████| 19/19 [00:00<00:00, 36.19it/s]


===== FINAL TEST RESULTS (FEDformer-Large) =====
Test MSE: 2.309434
Test MAE: 1.287613
